# Phase 1: DistilGPT-2 Decoder Smoke Tests

Zero-shot prompting on Jigsaw comments:
- **D1** — per-label yes/no questions
- **D2** — sentence completion (`This comment is ___`) for all 6 labels + argmax probe

Set `QUICK_MODE = True` in the setup cell for faster iteration (`20+20` eval sample).

In [ ]:
%matplotlib inline

import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

QUICK_MODE = False
THRESHOLD = 0.5

_cwd = Path.cwd()
_bootstrap = _cwd.parent if _cwd.name == "notebooks" else _cwd
sys.path.insert(0, str(_bootstrap / "src"))

from data import add_toxic_any, load_train_df, verify_data_files
from decoder import (
    D2_LABEL_CONTINUATIONS,
    build_d1_prompt,
    build_d2_prompt,
    d1_method_summary,
    d2_argmax_summary,
    d2_method_summary,
    predict_multilabel_d1,
    predict_multilabel_d2,
    score_yes_no,
)
from device import get_device, sync_if_cuda
from eval_sample import get_balanced_sample, get_smoke_texts, show_random_rows
from paths import (
    DECODER_SMOKE_SCORES_PATH,
    DISTILGPT2_MODEL_ID,
    LOCAL_DISTILGPT2_DIR,
    PROJECT_ROOT,
    SEED,
    TOXIC_LABELS,
    eval_n_each,
)

np.random.seed(SEED)
torch.manual_seed(SEED)

device = get_device()
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device} ({'GPU' if device.type == 'cuda' else 'CPU'})")
print(f"QUICK_MODE: {QUICK_MODE} (eval n_each={eval_n_each(QUICK_MODE)})")
print(f"PyTorch: {torch.__version__}")

## 1. Load data + balanced eval sample

In [ ]:
verify_data_files()
train_df = add_toxic_any(load_train_df())
sample = get_balanced_sample(train_df, quick_mode=QUICK_MODE)
texts = sample["comment_text"].astype(str).tolist()

print(f"Eval sample: {len(sample)} rows ({sample['toxic'].sum()} toxic, {(sample['toxic']==0).sum()} non-toxic)")

### Install model dependencies (run once)

If needed: `%pip install -r ../requirements-project.txt`, then restart kernel.

In [ ]:
# Uncomment and run once if needed:
# %pip install -r ../requirements-project.txt

## 2. Load DistilGPT-2

Prefetch: `python scripts/download_distilgpt2.py`

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_path = str(LOCAL_DISTILGPT2_DIR) if LOCAL_DISTILGPT2_DIR.exists() else DISTILGPT2_MODEL_ID
print(f"Loading from: {model_path}")

tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

## 3. D1 — yes/no prompting (smoke)

For **each of the 6 labels**, we ask the decoder an explicit question and score how it continues after `Answer:`.

**Example prompt (toxic):**
```
Is the following comment toxic? Answer yes or no.
Comment: {text}
Answer:
```

We compare four scoring paths:
- **logprob_full** — log P(` yes`) vs log P(` no`) over full tokens
- **logprob_first_token (C)** — first token only
- **constrained (D)** — greedy decode restricted to yes/no token ids
- **generate_parse (B)** — free generation; abstain if not parseable as yes/no

In [ ]:
print("D1 example prompt:")
print(build_d1_prompt(get_smoke_texts()[0], "toxic"))
print("-" * 60)

smoke_texts = get_smoke_texts()
smoke_d1 = predict_multilabel_d1(model, tokenizer, smoke_texts, device)
cols = ["comment_text", "d1_p_yes_toxic", "d1_pred_logprob_toxic", "d1_pred_constrained_toxic",
        "d1_pred_generate_toxic", "d1_parse_abstain_toxic", "d1_raw_completion_toxic"]
display(smoke_d1[cols].round(3))

## 4. D2 — sentence completion (smoke)

We ask the decoder to **complete the sentence** after:
```
Comment: {text}
This comment is ___
```

For each label we score a **positive** vs **negative** continuation (e.g. toxic: ` toxic` vs ` not`).
Same four methods as D1.

We also run an **argmax probe**: which label's positive phrase is the single most likely
next token? (mutually exclusive — diagnostic only, not true multi-label).

In [ ]:
print("D2 example prompt:")
print(build_d2_prompt(get_smoke_texts()[0]))
print("Continuation pairs per label:")
for lbl, (pos, neg) in D2_LABEL_CONTINUATIONS.items():
    print(f"  {lbl}: '{pos.strip()}' vs '{neg.strip()}'")
print("-" * 60)

smoke_d2 = predict_multilabel_d2(model, tokenizer, smoke_texts, device)
cols = ["comment_text", "d2_p_positive_toxic", "d2_pred_logprob_toxic", "d2_pred_constrained_toxic",
        "d2_pred_generate_toxic", "d2_parse_abstain_toxic", "d2_raw_completion_toxic", "d2_argmax_label"]
display(smoke_d2[cols].round(3))

## 5. D1 — balanced eval sample

6 prompts per comment. Slow on CPU — use `QUICK_MODE` while debugging.

In [ ]:
print(f"Running D1 on {len(texts)} comments x {len(TOXIC_LABELS)} labels ...")
d1_df = predict_multilabel_d1(model, tokenizer, texts, device)
show_random_rows(d1_df, n=5)

## 6. D2 — balanced eval sample

Same comment, 6 completion pairs + argmax label per row.

In [ ]:
print(f"Running D2 on {len(texts)} comments x {len(TOXIC_LABELS)} labels + argmax ...")
d2_df = predict_multilabel_d2(model, tokenizer, texts, device)
show_random_rows(d2_df, n=5)

## 7. D2 argmax — most likely single label

Picks the label whose positive continuation has highest first-step log-probability.
**Not multi-label** — at most one winner per comment.

In [ ]:
compare = sample[["comment_text"] + TOXIC_LABELS].copy()
compare = compare.merge(d1_df, on="comment_text").merge(d2_df, on="comment_text")

argmax_counts = compare["d2_argmax_label"].value_counts().rename_axis("argmax_label").reset_index(name="count")
display(argmax_counts)

argmax_diag = d2_argmax_summary(compare)
display(argmax_diag.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=compare, x="d2_argmax_label", order=TOXIC_LABELS, ax=ax, color="steelblue")
ax.set_title("D2 argmax label distribution (eval sample)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Sanity check — toxic label (D1 vs D2)

In [ ]:
compare["d1_p_toxic"] = compare["d1_p_yes_toxic"]
compare["d1_pred_toxic"] = compare["d1_pred_logprob_toxic"]
compare["d2_p_toxic"] = compare["d2_p_positive_toxic"]
compare["d2_pred_toxic"] = compare["d2_pred_logprob_toxic"]

for prefix, pred_col in [("D1 logprob", "d1_pred_toxic"), ("D2 logprob", "d2_pred_toxic")]:
    agree = (compare["toxic"] == compare[pred_col]).mean()
    print(f"{prefix} agreement vs gold toxic: {agree:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_df = compare.assign(gold_label=compare["toxic"].map({0: "non-toxic", 1: "toxic"}))
for ax, score_col, title in [
    (axes[0], "d1_p_toxic", "D1 P(yes) for toxic"),
    (axes[1], "d2_p_toxic", "D2 P(positive) for toxic"),
]:
    sns.violinplot(data=plot_df, x="gold_label", y=score_col, ax=ax, palette=["seagreen", "indianred"])
    ax.axhline(THRESHOLD, color="crimson", linestyle="--", linewidth=1)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 9. D1 scoring methods — agreement by label

In [ ]:
d1_method_df = d1_method_summary(compare)
display(d1_method_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=d1_method_df, x="label", y="agreement", hue="method", ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title("D1 agreement with gold by method")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")

abstain_df = d1_method_df[d1_method_df["method"] == "generate_parse"]
sns.barplot(data=abstain_df, x="label", y="abstain_rate", ax=axes[1], color="gray")
axes[1].set_ylim(0, 1)
axes[1].set_title("D1 generate_parse abstain rate")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 10. D2 scoring methods — agreement by label

In [ ]:
d2_method_df = d2_method_summary(compare)
display(d2_method_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=d2_method_df, x="label", y="agreement", hue="method", ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title("D2 agreement with gold by method")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")

abstain_df = d2_method_df[d2_method_df["method"] == "generate_parse"]
sns.barplot(data=abstain_df, x="label", y="abstain_rate", ax=axes[1], color="gray")
axes[1].set_ylim(0, 1)
axes[1].set_title("D2 generate_parse abstain rate")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 11. Scoring mismatches (logprob_full vs generate_parse)

In [ ]:
mismatch_rows = []
for label in TOXIC_LABELS:
    mismatch_rows.append({
        "label": label,
        "d1_mismatch": compare[f"d1_scoring_mismatch_{label}"].mean(),
        "d2_mismatch": compare[f"d2_scoring_mismatch_{label}"].mean(),
    })
mismatch_df = pd.DataFrame(mismatch_rows)
display(mismatch_df.round(3))

melted = mismatch_df.melt(id_vars="label", var_name="prompt_type", value_name="mismatch_rate")
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=melted, x="label", y="mismatch_rate", hue="prompt_type", ax=ax)
ax.set_title("logprob_full != generate_parse (non-abstain only)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 12. Inference micro-benchmark

In [ ]:
bench_text = texts[0]
bench_prompt = build_d1_prompt(bench_text, "toxic")
bench_repeats = 50 if QUICK_MODE else 200

for _ in range(3):
    score_yes_no(model, tokenizer, bench_prompt, device)
sync_if_cuda(device)

t0 = time.perf_counter()
for _ in range(bench_repeats):
    score_yes_no(model, tokenizer, bench_prompt, device)
sync_if_cuda(device)
ms_per_prompt = 1000 * (time.perf_counter() - t0) / bench_repeats

print(f"Model: {DISTILGPT2_MODEL_ID}")
print(f"Parameters: {n_params:,}")
print(f"Device: {device}")
print(f"D1 single prompt (1 label): {ms_per_prompt:.2f} ms")
print(f"D1 all 6 labels (estimate): {ms_per_prompt * 6:.2f} ms/comment")
print(f"D2 all 6 labels + argmax (estimate): {ms_per_prompt * 7:.2f} ms/comment")

## 13. Save decoder smoke scores

In [ ]:
out = compare.copy()
out["quick_mode"] = QUICK_MODE
out["model_id"] = DISTILGPT2_MODEL_ID
out["device"] = str(device)
out["threshold"] = THRESHOLD

DECODER_SMOKE_SCORES_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(DECODER_SMOKE_SCORES_PATH, index=False)
print(f"Saved {len(out)} rows to {DECODER_SMOKE_SCORES_PATH}")

## Next steps

- Join `encoder_smoke_scores.parquet` (notebook 01) with `decoder_smoke_scores.parquet`
- Deferred: formal eval notebook with train/val/test split
- Phase 2: `run_benchmark.py` — D3 few-shot, prompt ablations